# 🛍️ Shopping Mall Customer Segmentation
## Step 0 — Exploratory Data Analysis (EDA) & Data Pre-processing
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  
> ⚠️ **Run this notebook FIRST before any clustering notebooks.**  
> It will save a cleaned file `data_preprocessed.csv` and a scaled array `X_scaled.npy` for use by all members.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

print('✅ Libraries imported successfully!')

---
# PART 1 — Exploratory Data Analysis (EDA)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data/Shopping_Mall_Customer_Segmentation_Data_.csv')

print('Dataset Shape:', df.shape)
print(f'Total Customers: {len(df):,}')
print(f'Total Features : {df.shape[1]}')
print()
print('First 10 rows:')
df.head(10)

## 3. Dataset Overview

In [ ]:
print('─' * 45)
print('  COLUMN INFO')
print('─' * 45)
df.info()

In [ ]:
print('Statistical Summary:')
df.describe().round(2)

In [ ]:
print('Missing Values per Column:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print()
if missing.sum() == 0:
    print('✅ No missing values found!')
else:
    print(f'⚠️  Total missing values: {missing.sum()}')

In [ ]:
print('Duplicate Rows:', df.duplicated().sum())
print()
print('Gender Distribution:')
gender_counts = df['Gender'].value_counts()
gender_pct = df['Gender'].value_counts(normalize=True).mul(100).round(2)
gender_df = pd.DataFrame({'Count': gender_counts, 'Percentage %': gender_pct})
print(gender_df)

## 4. Distribution of Numeric Features

In [ ]:
numeric_cols = ['Age', 'Annual Income', 'Spending Score']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = ['steelblue', 'coral', 'mediumseagreen']

for ax, col, color in zip(axes, numeric_cols, colors):
    ax.hist(df[col], bins=35, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='red', linestyle=':', linewidth=1.5, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'Distribution of {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()
print('✅ Saved: eda_distributions.png')

## 5. Gender Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
gender_counts.plot.pie(ax=axes[0], autopct='%1.1f%%', colors=['#FF9999', '#66B2FF'],
                       startangle=90, explode=[0.03, 0.03], textprops={'fontsize': 12})
axes[0].set_title('Gender Distribution (Pie)', fontweight='bold')
axes[0].set_ylabel('')

# Bar chart by gender for each feature
gender_means = df.groupby('Gender')[numeric_cols].mean()
gender_means.T.plot(kind='bar', ax=axes[1], color=['#FF9999', '#66B2FF'], edgecolor='white')
axes[1].set_title('Average Feature Values by Gender', fontweight='bold')
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Average Value')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.savefig('eda_gender.png', bbox_inches='tight')
plt.show()
print('✅ Saved: eda_gender.png')

## 6. Box Plots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, col, color in zip(axes, numeric_cols, colors):
    df.boxplot(column=col, ax=ax, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'Box Plot — {col}', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Outlier Detection via Box Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_boxplots.png', bbox_inches='tight')
plt.show()
print('✅ Saved: eda_boxplots.png')

# IQR Outlier count
print('\nOutlier Count per Feature (IQR method):')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'  {col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)')

## 7. Scatter Plots — Feature Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [('Annual Income', 'Spending Score'), ('Age', 'Spending Score'), ('Age', 'Annual Income')]
gender_colors = {'Male': 'steelblue', 'Female': 'coral'}

for ax, (x_col, y_col) in zip(axes, pairs):
    for gender, group in df.groupby('Gender'):
        ax.scatter(group[x_col], group[y_col],
                   color=gender_colors[gender], label=gender, alpha=0.3, s=8)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{x_col} vs {y_col}', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Feature Relationships by Gender', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_scatter.png', bbox_inches='tight')
plt.show()
print('✅ Saved: eda_scatter.png')

## 8. Correlation Matrix

In [ ]:
df_corr = df[numeric_cols].copy()
corr_matrix = df_corr.corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, square=True,
            annot_kws={'size': 13})
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()

print('\nCorrelation Values:')
print(corr_matrix.round(4))
print('✅ Saved: eda_correlation.png')

## 9. Pairplot

In [ ]:
sample_df = df.sample(n=min(2000, len(df)), random_state=42)  # sample for speed
g = sns.pairplot(sample_df[numeric_cols + ['Gender']], hue='Gender',
                 palette={'Male': 'steelblue', 'Female': 'coral'},
                 diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
g.fig.suptitle('Pairplot of All Features (sample of 2000)', y=1.02, fontweight='bold')
plt.savefig('eda_pairplot.png', bbox_inches='tight')
plt.show()
print('✅ Saved: eda_pairplot.png')

---
# PART 2 — Data Pre-processing

## 10. Drop Irrelevant Column

In [ ]:
df_clean = df.drop(columns=['Customer ID'])

print('Columns before:', list(df.columns))
print('Columns after :', list(df_clean.columns))
print()
print('✅ Dropped: Customer ID (UUID — not useful for clustering)')

## 11. Encode Categorical Feature — Gender

In [ ]:
le = LabelEncoder()
df_clean['Gender'] = le.fit_transform(df_clean['Gender'])

print('Label Encoding mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls} → {i}')
print()
print('Sample after encoding:')
df_clean.head()

## 12. Handle Missing Values (if any)

In [ ]:
if df_clean.isnull().sum().sum() == 0:
    print('✅ No missing values — no imputation needed.')
else:
    print('⚠️ Missing values found — filling with column mean:')
    df_clean.fillna(df_clean.mean(numeric_only=True), inplace=True)
    print('✅ Done.')

print('\nFinal missing values check:')
print(df_clean.isnull().sum())

## 13. Feature Scaling — StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

print('Before Scaling:')
print(df_clean.describe().round(2))
print()

df_scaled_check = pd.DataFrame(X_scaled, columns=df_clean.columns)
print('After Scaling (mean ≈ 0, std ≈ 1):')
print(df_scaled_check.describe().round(4))

In [ ]:
# Visual comparison before vs after scaling
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
all_cols = df_clean.columns.tolist()

for i, col in enumerate(all_cols):
    axes[0, i].hist(df_clean[col], bins=30, color='coral', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col}\n(Before)', fontweight='bold')
    axes[0, i].set_xlabel('Value')

    axes[1, i].hist(df_scaled_check[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'{col}\n(After Scaling)', fontweight='bold')
    axes[1, i].set_xlabel('Scaled Value')

# Hide unused subplot
axes[0, len(all_cols)].set_visible(False)
axes[1, len(all_cols)].set_visible(False)

plt.suptitle('Feature Distributions: Before vs After StandardScaler', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing_scaling.png', bbox_inches='tight')
plt.show()
print('✅ Saved: preprocessing_scaling.png')

## 14. Save Preprocessed Data

In [ ]:
# Save cleaned (encoded but not scaled) dataframe
df_clean.to_csv('data_preprocessed.csv', index=False)

# Save scaled numpy array for clustering notebooks
np.save('X_scaled.npy', X_scaled)

print('✅ Saved: data_preprocessed.csv')
print('✅ Saved: X_scaled.npy')
print()
print('These files will be loaded directly by:')
print('  → member1_kmeans.ipynb')
print('  → member2_meanshift.ipynb')
print('  → member3_dbscan.ipynb')
print('  → comparison_all_methods.ipynb')

---
## 15. EDA & Preprocessing Summary

In [ ]:
print('=' * 55)
print('       EDA & PRE-PROCESSING SUMMARY')
print('=' * 55)
print(f'  Total Records          : {len(df):,}')
print(f'  Features Used          : {df_clean.shape[1]}')
print(f'  Missing Values         : None')
print(f'  Duplicate Rows         : {df.duplicated().sum()}')
print()
print('  Steps Performed:')
print('    1. Dropped Customer ID (irrelevant UUID)')
print('    2. Label Encoded Gender (Female=0, Male=1)')
print('    3. Handled missing values (none found)')
print('    4. Applied StandardScaler (mean=0, std=1)')
print()
print('  Output Files:')
print('    ✅ data_preprocessed.csv  (encoded, unscaled)')
print('    ✅ X_scaled.npy           (scaled, ready for clustering)')
print('=' * 55)
print()
print('▶ Ready! Proceed to the clustering notebooks.')